# Dasar Machine Learning

## Course 3 Bagian e: Bagging dan Random Forest

Data churn pelanggan telekomunikasi yang sudah diproses dipakai untuk membandingkan random forest dengan extra trees. Keduanya adalah metode bagging, yaitu banyak pohon dilatih pada sampel acak lalu hasilnya digabung.

### Tujuan

- Menilai pengaruh jumlah pohon terhadap out-of-bag error.
- Membandingkan random forest dengan extra randomized trees.
- Menghitung metrik klasifikasi dan menafsirkan kepentingan fitur.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns


## Bagian 1: Menyiapkan data


In [ ]:
DATA_URL = (
    "https://cf-courses-data.s3.us.cloud-object-storage.appdomain.cloud/"
    "IBM-ML241EN-SkillsNetwork/labs/datasets/churndata_processed.csv"
)
SAMPLE_SIZE = 3000

churn_data = pd.read_csv(DATA_URL).sample(
    n=SAMPLE_SIZE, random_state=42
).reset_index(drop=True)
churn_data.head()


In [ ]:
round(churn_data.describe().T, 2)


In [ ]:
figure, axis = plt.subplots(figsize=(12, 8))
sns.heatmap(churn_data.corr(), ax=axis)
plt.tight_layout()
plt.show()


## Bagian 2: Target dan pembagian data

Target `churn_value` tidak seimbang, sekitar 85 persen pelanggan tidak berhenti berlangganan. Pembagian data karena itu dibuat stratifikasi.


In [ ]:
target_name = "churn_value"
churn_data[target_name].value_counts(normalize=True)


In [ ]:
from sklearn.model_selection import StratifiedShuffleSplit

feature_columns = [name for name in churn_data.columns if name != target_name]
splitter = StratifiedShuffleSplit(n_splits=1, test_size=900, random_state=42)
train_index, test_index = next(
    splitter.split(churn_data[feature_columns], churn_data[target_name])
)

X_train = churn_data.loc[train_index, feature_columns]
y_train = churn_data.loc[train_index, target_name]
X_test = churn_data.loc[test_index, feature_columns]
y_test = churn_data.loc[test_index, target_name]

y_train.value_counts(normalize=True)


## Bagian 3: Random forest dan out-of-bag error

`warm_start=True` memungkinkan penambahan pohon tanpa melatih ulang dari awal. Out-of-bag error dihitung dari sampel yang tidak terpilih saat membangun setiap pohon.


In [ ]:
import warnings

from sklearn.ensemble import ExtraTreesClassifier, RandomForestClassifier

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=RuntimeWarning)

tree_counts = [15, 25, 50, 100, 200]
forest = RandomForestClassifier(oob_score=True, random_state=42,
                                warm_start=True, n_jobs=-1)

oob_rows = []
for n_trees in tree_counts:
    forest.set_params(n_estimators=n_trees).fit(X_train, y_train)
    oob_rows.append({"n_trees": n_trees, "oob": 1 - forest.oob_score_})

forest_oob = pd.DataFrame(oob_rows).set_index("n_trees")
forest_oob


In [ ]:
sns.set_context("talk")
sns.set_style("white")

axis = forest_oob.plot(legend=False, marker="o", figsize=(12, 6), linewidth=4)
axis.set(xlabel="Number of trees", ylabel="out-of-bag error")
plt.tight_layout()
plt.show()


## Bagian 4: Extra trees

`ExtraTreesClassifier` memilih titik potong secara acak, sehingga keragamannya lebih tinggi tetapi biasnya bisa lebih besar.


In [ ]:
extra_trees = ExtraTreesClassifier(oob_score=True, random_state=42,
                                   warm_start=True, bootstrap=True, n_jobs=-1)

extra_rows = []
for n_trees in tree_counts:
    extra_trees.set_params(n_estimators=n_trees).fit(X_train, y_train)
    extra_rows.append({"n_trees": n_trees, "oob": 1 - extra_trees.oob_score_})

extra_oob = pd.DataFrame(extra_rows).set_index("n_trees")

oob_comparison = pd.concat(
    [
        forest_oob.rename(columns={"oob": "RandomForest"}),
        extra_oob.rename(columns={"oob": "ExtraTrees"}),
    ],
    axis=1,
)
oob_comparison


In [ ]:
axis = oob_comparison.plot(marker="o", figsize=(12, 6), linewidth=4)
axis.set(xlabel="Number of trees", ylabel="out-of-bag error")
plt.tight_layout()
plt.show()


## Bagian 5: Metrik hasil

Model dengan 100 pohon dipakai untuk menilai kinerja pada data uji.


In [ ]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)

final_model = RandomForestClassifier(
    n_estimators=100, random_state=42, n_jobs=-1
).fit(X_train, y_train)
predictions = final_model.predict(X_test)

print(classification_report(y_test, predictions))
pd.DataFrame(
    {
        "accuracy": accuracy_score(y_test, predictions),
        "precision": precision_score(y_test, predictions),
        "recall": recall_score(y_test, predictions),
        "f1": f1_score(y_test, predictions),
        "auc": roc_auc_score(y_test, predictions),
    },
    index=[0],
)


## Bagian 6: Menelaah hasil

Recall untuk pelanggan yang berhenti berlangganan relatif rendah karena jumlah kelasnya kecil. Confusion matrix, kurva ROC, kurva precision-recall, dan kepentingan fitur membantu memahami pola ini.


In [ ]:
from sklearn.metrics import ConfusionMatrixDisplay, confusion_matrix

sns.set_context("talk")
matrix = confusion_matrix(y_test, predictions, normalize="true")
ConfusionMatrixDisplay(
    confusion_matrix=matrix, display_labels=final_model.classes_
).plot()
plt.show()


In [ ]:
from sklearn.metrics import precision_recall_curve, roc_curve

probabilities = final_model.predict_proba(X_test)[:, 1]
false_positive, true_positive, _ = roc_curve(y_test, probabilities)
precision, recall, _ = precision_recall_curve(y_test, probabilities)

figure, axes = plt.subplots(ncols=2, figsize=(14, 6))

axes[0].plot(false_positive, true_positive, linewidth=4)
axes[0].plot([0, 1], [0, 1], ls="--", color="black", lw=.5)
axes[0].set(xlabel="False Positive Rate", ylabel="True Positive Rate",
            xlim=[-.01, 1.01], ylim=[-.01, 1.01], title="ROC curve")
axes[0].grid(True)

axes[1].plot(recall, precision, linewidth=4)
axes[1].set(xlabel="Recall", ylabel="Precision",
            xlim=[-.01, 1.01], ylim=[-.01, 1.01],
            title="Precision-Recall curve")
axes[1].grid(True)

plt.tight_layout()
plt.show()


In [ ]:
feature_importance = pd.Series(
    final_model.feature_importances_, index=feature_columns
).sort_values(ascending=False)

figure, axis = plt.subplots(figsize=(14, 6))
axis.pie(feature_importance, labels=None,
         autopct=lambda pct: f"{pct:1.1f}%" if pct > 5.5 else "")
axis.set(ylabel="Relative Importance", xlabel="Feature")
plt.legend(loc="center left", bbox_to_anchor=(1, 0.5),
           labels=feature_importance.index)
plt.tight_layout()
plt.show()


## Ringkasan

Random forest memberi out-of-bag error lebih rendah daripada extra trees pada data ini. Kepuasan pelanggan dan jenis kontrak merupakan prediktor churn paling kuat, sedangkan recall kelas churn tetap terbatas karena ketidakseimbangan kelas.
